# EDA — Nanobodies unidos a proteína (HER2 marcado) · P1

Este cuaderno lee una versión ya cerrada del dataset (`data/releases/<versión>/`) y
la describe con números y gráficos. Antes hay que haber corrido los scripts 01 a 05.

El dataset son todos los nanobodies (VHH) del PDB unidos a una proteína. La columna
`is_her2` marca los que se unen a HER2/ERBB2. Varias gráficas separan **HER2** del
**resto** para que veas dónde cae tu diana dentro del conjunto.

Cambia `VERSION` si quieres mirar otra versión. Solo necesita `pandas` y `matplotlib`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

VERSION = 'v1.0'
BASE = Path('..')
RELEASE_DIR = BASE / 'data' / 'releases' / VERSION


def leer_fasta(ruta):
    """Devuelve {nombre: secuencia}."""
    secuencias, nombre = {}, None
    for linea in open(ruta, encoding='utf-8'):
        linea = linea.strip()
        if linea.startswith('>'):
            nombre = linea[1:].split()[0]
            secuencias[nombre] = ''
        elif nombre:
            secuencias[nombre] += linea
    return secuencias


dataset = pd.read_csv(RELEASE_DIR / 'dataset.tsv', sep='\t')
es_her2 = dataset['is_her2'].astype(str) == 'True'
print(f'{len(dataset)} nanobodies representantes | de ellos HER2: {es_her2.sum()}')
RELEASE_DIR

## 1. Cuántas filas sobreviven a cada paso

Es la versión en números de lo que está escrito en `DECISIONS.md`. La columna `n_her2`
dice cuántos de los que sobreviven son anti-HER2.

In [ ]:
filter_log = pd.read_csv(RELEASE_DIR / 'filter_log.csv')
filter_log

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(filter_log['paso'][::-1], filter_log['n_entradas'][::-1])
ax.set_xlabel('Filas restantes')
ax.set_title('Filas tras cada paso del pipeline')
plt.tight_layout()
plt.show()

**Qué mirar:** el salto entre `nanobody_SD-H` y `antigeno_proteina` es cuántos nanobodies
no estaban unidos a una proteína. Si un filtro se lleva casi todo, vuelve a
`DECISIONS.md` y pregúntate si el umbral tiene sentido.

## 2. HER2 dentro del conjunto

Cuántos representantes finales son de HER2 y cuántos del resto. Es poco: por eso el
dataset es "todos los nanobodies", no solo HER2. Aun así HER2 queda localizado.

In [ ]:
conteo = es_her2.value_counts().rename({True: 'HER2', False: 'resto'})
fig, ax = plt.subplots(figsize=(5, 3.5))
conteo.plot(kind='bar', ax=ax, color=['tab:orange', 'tab:blue'])
ax.set_ylabel('Nº de nanobodies representantes')
ax.set_title('HER2 vs resto en el dataset final')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print('Representantes anti-HER2:')
cols = [c for c in ['pdb_id_4', 'rcsb_resolution', 'CDR-H3', 'antigen_name'] if c in dataset.columns]
print(dataset.loc[es_her2, cols].to_string(index=False))

## 3. Antígenos más frecuentes

Contra qué proteínas hay más nanobodies resueltos. Da una idea de qué dianas están más
estudiadas (y de dónde hay más datos para entrenar en el futuro).

In [ ]:
top = dataset['antigen_name'].fillna('sin dato').str.strip().value_counts().head(15)
fig, ax = plt.subplots(figsize=(8, 5))
top[::-1].plot(kind='barh', ax=ax)
ax.set_xlabel('Nº de nanobodies representantes')
ax.set_title('15 antígenos con más nanobodies')
plt.tight_layout()
plt.show()

## 4. Longitud del dominio variable (HER2 vs resto)

In [ ]:
dataset['len_vh'] = dataset['VH'].str.len()
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist([dataset.loc[~es_her2, 'len_vh'], dataset.loc[es_her2, 'len_vh']],
        bins=range(100, 142, 3), stacked=True, label=['resto', 'HER2'],
        color=['tab:blue', 'tab:orange'])
ax.set_xlabel('Longitud (aminoácidos)')
ax.set_ylabel('Nº de nanobodies')
ax.set_title('Longitud del dominio VHH')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Especie de origen

Los nanobodies suelen venir de camélidos inmunizados (llama, alpaca, dromedario) o de
librerías sintéticas. Si casi todos vienen de un mismo origen, el dataset está sesgado.

In [ ]:
col_especie = 'heavy_species' if 'heavy_species' in dataset.columns else 'organism'
conteo = dataset[col_especie].fillna('sin dato').value_counts().head(12)

fig, ax = plt.subplots(figsize=(7, 5))
conteo[::-1].plot(kind='barh', ax=ax)
ax.set_xlabel('Nº de nanobodies')
ax.set_title('Especie de origen del nanobody (top 12)')
plt.tight_layout()
plt.show()

## 6. Resolución de las estructuras

In [ ]:
pdb_meta = pd.read_csv(BASE / 'data' / 'raw' / 'pdb_metadata.csv')
con_resolucion = pdb_meta.dropna(subset=['resolution'])

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(con_resolucion['resolution'], bins=20)
ax.axvline(3.0, color='red', linestyle='--', label='corte usado (3.0 Å)')
ax.set_xlabel('Resolución (Å)')
ax.set_ylabel('Nº de estructuras')
ax.set_title('Resolución de todas las estructuras candidatas (antes de filtrar)')
ax.legend()
plt.tight_layout()
plt.show()

print('Método experimental:')
print(pdb_meta['method'].value_counts())

## 7. Efecto de la deduplicación

Cada grupo son nanobodies casi idénticos (≥ 95 % de identidad). Un grupo grande
significa que esa secuencia estaba muy repetida en el PDB.

In [ ]:
clusters = pd.read_csv(RELEASE_DIR / 'cluster_assignments.tsv', sep='\t')
tamanos = clusters.groupby('cluster_id').size()

print(f'Filas antes de deduplicar: {len(clusters)}')
print(f'Grupos (secuencias distintas): {len(tamanos)}')
print(f'Reducción: {100 * (1 - len(tamanos) / len(clusters)):.1f} %')

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(tamanos, bins=range(1, int(tamanos.max()) + 2), align='left')
ax.set_xlabel('Tamaño del grupo (nº de estructuras)')
ax.set_ylabel('Nº de grupos')
ax.set_yscale('log')
ax.set_title('Cuántas copias casi idénticas tenía cada nanobody')
plt.tight_layout()
plt.show()

**Limitación (ver `DECISIONS.md`, apartado 3):** algunos grupos juntan secuencias con
CDR-H3 distinta. La columna `cluster_n_cdr_h3` del `dataset.tsv` dice cuántas CDR-H3
distintas hay en cada grupo; los que valen más de 1 son los que habría que revisar.

In [ ]:
if 'cluster_n_cdr_h3' in dataset.columns:
    mezclados = (dataset['cluster_n_cdr_h3'] > 1).sum()
    print(f'Grupos con más de una CDR-H3 distinta: {mezclados} de {len(dataset)}')

## 8. Año de depósito

Sirve para ver si los nanobodies son un área activa: si la mayoría son recientes, hay
cada vez más datos disponibles.

In [ ]:
pdb_meta['año'] = pd.to_datetime(pdb_meta['deposit_date'], errors='coerce', utc=True).dt.year
por_año = pdb_meta['año'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(por_año.index.astype(int).astype(str), por_año.values)
ax.set_xlabel('Año de depósito en el PDB')
ax.set_ylabel('Nº de estructuras')
ax.set_title('Estructuras candidatas por año de depósito')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## 9. Longitud de la CDR-H3

La CDR-H3 es la zona más variable del nanobody y la que más decide a qué se une.
Aquí se ve su longitud en los representantes finales.

In [ ]:
if 'CDR-H3' in dataset.columns:
    longitud_cdr3 = dataset['CDR-H3'].dropna().str.len()
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(longitud_cdr3, bins=range(0, int(longitud_cdr3.max()) + 3, 2))
    ax.set_xlabel('Longitud de la CDR-H3 (aminoácidos)')
    ax.set_ylabel('Nº de nanobodies')
    ax.set_title('Longitud de la CDR-H3')
    plt.tight_layout()
    plt.show()
else:
    print('La tabla no tiene columna CDR-H3.')

## 10. Resumen (rellénalo tú, con tus palabras)

- Nº de nanobodies en el dataset final: ...
- ¿Cuántos son de HER2?: ...
- ¿Cuántos falsos nanobodies se quitaron en el paso 01?: ...
- ¿Qué origen (especie) predomina?: ...
- ¿Cuánto redujo la deduplicación?: ...
- ¿Qué antígenos son los más frecuentes?: ...
- Qué umbral cambiarías en la siguiente versión y por qué: ...